# Circular Finned Tubes - Dry Air-Cooler Example (v0.7.x, experimental)

This notebook uses the **real property providers** and the **real solver**
(`BareTubeHeatExchanger.solve`/`.simulate`) -- no notebook-local correlations.

It demonstrates the v0.7.x experimental circular finned-tube feature
(`experiment/v0.7.x-finned-tubes-claude`, not merged into `main`, not released):

1. A **welded** constant-thickness finned tube (`D_root == D_o`)
2. An **extruded** root-to-tip tapered finned tube (`D_root > D_o`)
3. Geometry results (areas, volume, fin-blockage-aware V_max)
4. Fin efficiency (closed-form vs. numerical tapered solver)
5. Briggs & Young (1963) outside heat-transfer coefficient
6. The Robinson & Briggs (1966) pressure-drop **documented blocker**
7. A complete dry `BareTubeHeatExchanger.simulate()` run

See `docs/finned_tube_model.md` for the full write-up, definitions, and
unresolved limitations.

In [1]:
from pathlib import Path
import sys

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

workspace_root = None
for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break

if workspace_root is None:
    raise RuntimeError("Could not locate the repository root (a parent containing 'core/').")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("workspace_root =", workspace_root)

workspace_root = C:\Users\pawel\GitHub\kalkalori


In [2]:
import math
import pandas as pd

from core.geometry import BareTube, CircularFinnedTube, TubeBundle
from core.geometry.finned_flow_geometry import finned_vmax_ratio_min_freeflow
from core.heat_transfer.fin_efficiency import (
    fin_efficiency_constant_thickness,
    fin_efficiency_tapered,
    overall_surface_efficiency,
)
from core.heat_transfer.finned_tube_resistance import build_finned_tube_resistance_network
from core.heat_transfer.outside_flow import FluidProps as OutsideFluidProps
from core.heat_transfer.outside_flow_finned import (
    HTC_CONTRACT,
    finned_outside_flow_from_mass_flow,
    nusselt_briggs_young,
)
from core.heat_transfer.internal_flow import FluidProps as InternalFluidProps
from core.heat_transfer.streams import SensibleHeatStream
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput
from core.properties.dry_air import DryAirPropertyProvider
from core.pressure_drop.outside_pressure_drop import RobinsonBriggsEulerProvider, EulerRequest


## 1. Geometry: welded (constant thickness) vs. extruded (tapered) tube

In [3]:
core_tube = BareTube(
    D_i=0.0180, D_o=0.0254, length_total=3.0, length_effective=3.0, wall_k=45.0,
)

welded_tube = CircularFinnedTube(
    core_tube=core_tube,
    fin_k=200.0,             # aluminum-like fin conductivity [W/(m*K)]
    D_fin=0.0570,
    D_root=core_tube.D_o,    # welded: no separate foot layer
    fin_thickness_root=0.00040,
    fin_pitch=0.00230,       # ~11 fins/inch pitch
    fin_contact_resistance=0.0,  # explicitly ideal contact (welded fin)
)

extruded_tube = CircularFinnedTube(
    core_tube=core_tube,
    fin_k=200.0,
    D_fin=0.0570,
    D_root=0.0286,           # integral foot layer, D_root > D_o
    fin_thickness_root=0.00060,
    fin_thickness_tip=0.00020,   # root-to-tip taper
    fin_pitch=0.00230,
    fin_contact_resistance=None, # unknown -> ideal fallback + warning
)

print(welded_tube.describe())
print(extruded_tube.describe())

CircularFinnedTube(D_i=0.018, D_o=0.0254, D_root=0.0254, D_fin=0.057, fin_pitch=0.0023, fin_thickness_root=0.0004, fin_thickness_tip=0.0004)
CircularFinnedTube(D_i=0.018, D_o=0.0254, D_root=0.0286, D_fin=0.057, fin_pitch=0.0023, fin_thickness_root=0.0006, fin_thickness_tip=0.0002)


In [4]:
def geometry_row(name, tube):
    return {
        "tube": name,
        "D_root [mm]": tube.D_root * 1e3,
        "D_fin [mm]": tube.D_fin * 1e3,
        "fin_height [mm]": tube.fin_height * 1e3,
        "root_radial_thickness [mm]": tube.root_radial_thickness * 1e3,
        "fin_density [1/m]": tube.fin_density,
        "effective_fin_count [-]": tube.effective_fin_count,
        "clear_spacing_root [mm]": tube.clear_spacing_root * 1e3,
        "A_primary [m^2]": tube.A_primary,
        "A_fin [m^2]": tube.A_fin,
        "A_outside_geometric [m^2]": tube.A_outside_geometric,
        "A_outside_gross [m^2]": tube.A_outside_gross,
        "area_outer_to_bare_ratio [-]": tube.area_outer_to_bare_ratio,
        "fin_volume_per_length [m^3/m]": tube.fin_volume_per_length,
    }

geometry_df = pd.DataFrame([geometry_row("welded (const. t)", welded_tube), geometry_row("extruded (tapered)", extruded_tube)])
geometry_df.set_index("tube").T

tube,welded (const. t),extruded (tapered)
D_root [mm],25.400000,28.600000
D_fin [mm],57.000000,57.000000
fin_height [mm],15.800000,14.200000
root_radial_thickness [mm],0.000000,1.600000
fin_density [1/m],434.782609,434.782609
effective_fin_count [-],1304.347826,1304.347826
clear_spacing_root [mm],1.900000,1.700000
A_primary [m^2],0.197756,0.199232
A_fin [m^2],5.428344,5.028080
A_outside_geometric [m^2],5.626101,5.227312


## 2. Fin-blockage-aware minimum free-flow area / V_max

`finned_vmax_ratio_min_freeflow` generalizes the existing bare-tube
`vmax_ratio_min_freeflow` by accounting for periodic blockage from the
root **and** the fins (not the bare root diameter alone).

In [5]:
S_T, S_L = 0.0650, 0.0550   # staggered (triangular pitch) transverse/longitudinal spacing

ratio_welded = finned_vmax_ratio_min_freeflow(welded_tube, S_T, S_L, "staggered")
ratio_extruded = finned_vmax_ratio_min_freeflow(extruded_tube, S_T, S_L, "staggered")
print(f"V_max/V_face ratio (welded)  : {ratio_welded:.4f}")
print(f"V_max/V_face ratio (extruded): {ratio_extruded:.4f}")

V_max/V_face ratio (welded)  : 1.9059
V_max/V_face ratio (extruded): 2.0661


## 3. Fin efficiency: closed form vs. numerical tapered solver

In [6]:
h_o_assumed = 80.0  # W/(m^2*K), representative outside HTC for this demonstration

eta_welded = fin_efficiency_constant_thickness(
    D_root=welded_tube.D_root, D_fin=welded_tube.D_fin,
    fin_thickness=welded_tube.fin_thickness_root, fin_k=welded_tube.fin_k, h_o=h_o_assumed,
)
tapered_solution = fin_efficiency_tapered(
    D_root=extruded_tube.D_root, D_fin=extruded_tube.D_fin,
    fin_thickness_root=extruded_tube.fin_thickness_root,
    fin_thickness_tip=extruded_tube.fin_thickness_tip_used,
    fin_k=extruded_tube.fin_k, h_o=h_o_assumed, n_steps=2000,
)

# Cross-check: the numerical tapered solver reduces to the closed form
# in the constant-thickness limit (independent verification, not a
# tautology -- see core/tests/finned_tube_fin_efficiency_test.py).
cross_check = fin_efficiency_tapered(
    D_root=welded_tube.D_root, D_fin=welded_tube.D_fin,
    fin_thickness_root=welded_tube.fin_thickness_root,
    fin_thickness_tip=welded_tube.fin_thickness_root,
    fin_k=welded_tube.fin_k, h_o=h_o_assumed, n_steps=2000,
).efficiency

print(f"eta_fin welded (closed form)          : {eta_welded:.4f}")
print(f"eta_fin welded (numerical, same case) : {cross_check:.4f}  (cross-check)")
print(f"eta_fin extruded/tapered (numerical)  : {tapered_solution.efficiency:.4f}")

eta_o_welded = overall_surface_efficiency(A_primary=welded_tube.A_primary, A_fin=welded_tube.A_fin_used, fin_efficiency=eta_welded)
eta_o_extruded = overall_surface_efficiency(A_primary=extruded_tube.A_primary, A_fin=extruded_tube.A_fin_used, fin_efficiency=tapered_solution.efficiency)
print(f"overall surface efficiency welded : {eta_o_welded:.4f}")
print(f"overall surface efficiency extruded: {eta_o_extruded:.4f}")

eta_fin welded (closed form)          : 0.8001
eta_fin welded (numerical, same case) : 0.8001  (cross-check)
eta_fin extruded/tapered (numerical)  : 0.8658
overall surface efficiency welded : 0.8071
overall surface efficiency extruded: 0.8709


## 4. Resistance network

Explicit inside convection / core-tube wall / root-layer conduction / contact
resistance / parallel primary+fin convection -- see `docs/finned_tube_model.md` section 5.

In [7]:
alfa_i_assumed = 300.0  # W/(m^2*K), representative inside (tube-side) HTC
n_tubes = 40

network_welded, warnings_welded = build_finned_tube_resistance_network(
    welded_tube, n_tubes=n_tubes, alfa_i=alfa_i_assumed, alfa_o_physical=h_o_assumed,
)
network_extruded, warnings_extruded = build_finned_tube_resistance_network(
    extruded_tube, n_tubes=n_tubes, alfa_i=alfa_i_assumed, alfa_o_physical=h_o_assumed,
)

resistance_df = pd.DataFrame([
    {
        "tube": "welded",
        "R_inside [K/W]": network_welded.R_inside_convection,
        "R_wall [K/W]": network_welded.R_wall_conduction,
        "R_root [K/W]": network_welded.R_root_conduction,
        "R_contact [K/W]": network_welded.R_contact,
        "R_outside [K/W]": network_welded.R_outside_convection,
        "UA [W/K]": network_welded.UA,
    },
    {
        "tube": "extruded",
        "R_inside [K/W]": network_extruded.R_inside_convection,
        "R_wall [K/W]": network_extruded.R_wall_conduction,
        "R_root [K/W]": network_extruded.R_root_conduction,
        "R_contact [K/W]": network_extruded.R_contact,
        "R_outside [K/W]": network_extruded.R_outside_convection,
        "UA [W/K]": network_extruded.UA,
    },
])
resistance_df.set_index("tube").T

tube,welded,extruded
R_inside [K/W],0.000491,4.912190e-04
R_wall [K/W],0.000010,1.014988e-05
R_root [K/W],0.000000,7.868723e-07
R_contact [K/W],0.000000,0.000000e+00
R_outside [K/W],0.000069,6.864056e-05
UA [W/K],1753.811595,1.751939e+03


In [8]:
print("Warnings for the extruded tube (unspecified contact resistance):")
for w in warnings_extruded:
    print(f"  [{w.severity}] {w.code}: {w.message}")
print()
print("Warnings for the welded tube (explicit ideal contact -> none expected):")
for w in warnings_welded:
    print(f"  [{w.severity}] {w.code}: {w.message}")

Warnings for the extruded tube (unspecified contact resistance):
  [warning] finned_tube_contact_resistance_unknown: finned_tube_resistance: fin_contact_resistance was not supplied; assuming ideal (zero) contact between the core tube and the fin root/foot. Supply an explicit value (0.0 for genuinely ideal contact) to remove this warning.

Warnings for the welded tube (explicit ideal contact -> none expected):


## 5. Briggs & Young (1963) outside heat-transfer coefficient

In [9]:
air_props = OutsideFluidProps(rho=1.15, mu=1.9e-5, k=0.028, cp=1007.0)

htc_result = finned_outside_flow_from_mass_flow(
    m_dot=5.0, frontal_area=2.0, tube=welded_tube,
    tube_pitch_transverse=S_T, tube_pitch_longitudinal=S_L,
    layout="staggered", n_rows=4, n_tubes_per_row=10,
    props=air_props, calculate_pressure_drop=True,
)

print("Correlation contract:", HTC_CONTRACT)
print()
print(f"face velocity v        : {htc_result.v:.3f} m/s")
print(f"V_max                   : {htc_result.V_max:.3f} m/s")
print(f"Re (root-diameter basis): {htc_result.Re:.1f}")
print(f"Pr                      : {htc_result.Pr:.4f}")
print(f"Nu (Briggs-Young)       : {htc_result.Nu:.3f}")
print(f"alfa_o physical         : {htc_result.alfa_o_physical:.2f} W/(m^2*K)")
print(f"pressure drop available : {htc_result.dp_available}")
print(f"dp_o                    : {htc_result.dp_o}")

Correlation contract: FinnedCorrelationContract(method='briggs_young_1963', source='Briggs, D.E.; Young, E.H. (1963), "Convection Heat Transfer and Pressure Drop of Air Flowing across Triangular Pitch Banks of Finned Tubes", Chemical Engineering Progress Symposium Series, Vol. 59, No. 41, pp. 1-10.', geometry_family='circular_finned_tube_bank', velocity_basis='maximum_gap_velocity_fin_blockage_aware', reynolds_basis='root_diameter_Vmax', reference_diameter='D_root', area_basis='physical_finned_surface_no_fin_efficiency', row_basis='not_row_resolved_0D')

face velocity v        : 2.174 m/s
V_max                   : 4.143 m/s
Re (root-diameter basis): 6369.8
Pr                      : 0.6833
Nu (Briggs-Young)       : 35.921
alfa_o physical         : 39.60 W/(m^2*K)
pressure drop available : False
dp_o                    : nan


## 6. The Robinson & Briggs (1966) pressure-drop blocker

This is a **documented, geometry-gated blocker**, not a silent substitute:
the exact closed-form Euler/friction-factor equation could not be
independently verified from accessible sources in this pass (see
`docs/finned_tube_model.md`, "Unresolved limitations"). The provider
validates geometry/layout and then explicitly raises `NotImplementedError`.

In [10]:
finned_request = EulerRequest(Re=5000.0, ST_over_D=2.6, SL_over_D=2.2, layout="staggered", n_rows=4, is_finned=True)
try:
    RobinsonBriggsEulerProvider().evaluate(finned_request)
except NotImplementedError as exc:
    print("NotImplementedError (expected, documented blocker):")
    print(" ", exc)

NotImplementedError (expected, documented blocker):
  Robinson-Briggs (1966) finned-tube pressure-drop model is not implemented in GPL core: the exact closed-form equation could not be independently verified from accessible sources in this pass (see docs/finned_tube_model.md, 'Unresolved limitations'). This provider name is reserved for a future contributor with primary-source access to Chem. Eng. Prog. Symp. Ser. 62(64), pp. 177-184 (1966).


In [11]:
print("Warnings from the full HTC+dp call above (dp explicitly unavailable, HTC still usable):")
for w in htc_result.warnings:
    print(f"  [{w.severity}] {w.code}")

Warnings from the full HTC+dp call above (dp explicitly unavailable, HTC still usable):
  [warning] outside_dp_few_rows
  [critical] outside_dp_robinson_briggs_not_implemented
  [critical] outside_dp_finned_unavailable


## 7. Complete dry Simulation

`BareTubeHeatExchanger.simulate()` on a bundle of welded finned tubes,
using the real solver end to end (no notebook-local shortcuts).

In [12]:
bundle = TubeBundle(
    tube=welded_tube,
    n_rows=4, n_tubes_per_row=10,
    pitch_transverse=S_T, pitch_longitudinal=S_L,
    layout="staggered", n_passes_tube=1, flow_arrangement="crossflow",
)
hx = BareTubeHeatExchanger(bundle=bundle)

inside = HXSideInput(provider=DryAirPropertyProvider(), m_dot=1.5, T_in=450.0, p=101_325.0)
outside = HXSideInput(provider=DryAirPropertyProvider(), m_dot=5.0, T_in=300.0, p=101_325.0)

sim = hx.simulate(inside, outside, flow_arrangement="crossflow")

print(f"surface_type          : {bundle.tube.surface_type}")
print(f"Q                     : {sim.q / 1e3:.2f} kW")
print(f"T_out (tube side)     : {sim.T_out_inside:.2f} K")
print(f"T_out (outside/air)   : {sim.T_out_outside:.2f} K")
print(f"UA                    : {sim.final_result.UA:.1f} W/K")
print(f"alfa_o (gross basis)  : {sim.final_result.outside_side_thermal.alfa:.2f} W/(m^2*K)")
print(f"A_o (gross, used)     : {sim.final_result.A_o:.3f} m^2")
print(f"outside Re            : {sim.final_result.outside_side_hydraulic.Re:.1f}")
print(f"outside dp_total      : {sim.final_result.outside_side_hydraulic.dp_total} (NaN: Robinson-Briggs blocker, see section 6)")

print()
print("Warnings:")
for w in (sim.final_result.warnings or []):
    print(f"  [{w.severity}] {w.code}")

surface_type          : TubeSurfaceType.CIRCULAR_FINNED
Q                     : 147.09 kW
T_out (tube side)     : 353.33 K
T_out (outside/air)   : 329.21 K
UA                    : 1874.0 W/K
alfa_o (gross basis)  : 35.34 W/(m^2*K)
A_o (gross, used)     : 225.044 m^2
outside Re            : 6454.0
outside dp_total      : nan (NaN: Robinson-Briggs blocker, see section 6)

Warnings:
  [info] tube_bundle_hydraulics_midpoint_temperature_fallback
  [info] tube_bundle_hydraulics_pass_boundary_temperature_fallback
  [warning] outside_dp_few_rows
  [critical] outside_dp_robinson_briggs_not_implemented
  [critical] outside_dp_finned_unavailable


## 8. Wet-surface guard (controlled rejection)

Wet/condensing finned surfaces are out of scope for this feature. Attempting
to use a phase-change-capable outside provider on a `CircularFinnedTube`
raises a controlled error rather than silently running (or silently
skipping) condensation physics.

In [13]:
from core.models.bare_tube import FinnedTubeWetOutsideSurfaceNotSupportedError
from core.properties.water import IAPWS97WaterSteamProvider

outside_wet = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=5.0, T_in=380.0, p=101_325.0)
try:
    hx.simulate(inside, outside_wet, flow_arrangement="crossflow")
except FinnedTubeWetOutsideSurfaceNotSupportedError as exc:
    print("FinnedTubeWetOutsideSurfaceNotSupportedError (expected):")
    print(" ", str(exc)[:220], "...")

FinnedTubeWetOutsideSurfaceNotSupportedError (expected):
  Wet/condensing finned surfaces are out of scope for CircularFinnedTube in this experimental pass (v0.7.x): the outside provider (IAPWS97WaterSteamProvider) is phase-change capable (component='H2O'). Use a non-condensing  ...


## Summary

- Geometry, fin efficiency, and the resistance network are internally
  cross-validated (closed-form vs. numerical solver; areas vs. direct
  numerical quadrature) and produce physically sensible, finite results.
- The Briggs & Young (1963) outside HTC is fully functional for staggered
  circular finned-tube banks.
- The Robinson & Briggs (1966) pressure drop is an honest, documented
  blocker: `dp_total` is `NaN` with explicit critical warnings, never a
  silently wrong number and never a bare-tube correlation reused outside
  its scope.
- Wet/condensing finned surfaces are explicitly rejected.

See `docs/finned_tube_model.md` for the full model description, definitions
table, and unresolved limitations.